# Loss and metrics analysis

In this notebook we analyse the metrics captured during the experiments and verify that the reparam checkpoints differ from the baseline checkpoints in terms of loss landscape and generalization performance.

## Method of experimentation

Dataset: CIFAR-10
Models: ResNet-18 and ViT-B/32
Optimizers: SGD, SAM, ASAM, M-SAM

We train ResNet-18 model on CIFAR-10 for 200 epochs with SGD, SAM, ASAM, and M-SAM optimizers and fine-tune ViT-B/32 on CIFAR-10 for 50 epochs with the same optimizers. During the training, we capture following metrics:

- Model checkpoint
- training loss and accuracy
- test loss and accuracy
- seed
- optimizer type and hyperparameters (e.g. rho for SAM)

### Statistical analysis of metrics

We track divergence rate $\delta = \mathcal{L}_{test} - \mathcal{L}_{train}$ and Test accuracy against the hyperparameter $\rho$ for SAM-based optimizers. 
$\rho$ controls the size of the neighborhood considered for sharpness-aware optimization, and we expect that higher $\rho$ values will lead to flatter minima and thus lower divergence rates. We will analyze the relationship between $\delta$ and $\rho$ to verify this hypothesis.
However, we also note that the relationship between $\delta$ and $\rho$ may not be strictly monotonic, as excessively high $\rho$ values can lead to optimization instability and worse generalization.

In this experiment we sweep $\rho$ for SAM-based optimizers in the range $\{0.05, 0.1, 0.5\}$ and track the corresponding $\delta$ values to analyze this relationship.

In [5]:
import os
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch

seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(seed)

ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path("/Users/manavdahra/workspace/sam-opt-research")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / "results"

METRICS_PATHS = {
    "resnet18": {
        "baseline": RESULTS_DIR / "resnet18/experiments/baseline/resnet18/baseline_results.json",
        "reparam":  RESULTS_DIR / "resnet18/experiments/reparam/resnet18/reparam_results.json",
    },
    "vit_b_32": {
        "baseline": RESULTS_DIR / "vit_b_32/experiments/baseline/vit_b_32/baseline_results.json",
        "reparam":  RESULTS_DIR / "vit_b_32/experiments/reparam/vit_b_32/reparam_results.json",
    },
}


### Analyse the metrics

We take Resnet-18 with SGD optimizer as our baseline model. We compare the training and test losses of the baseline model with those of the reparametrized models to verify that the reparametrization leads to flatter minima and better generalization. We also analyze the relationship between $\delta$ and $\rho$ for SAM-based optimizers to understand how the choice of $\rho$ affects the generalization performance of the model.

In [6]:
import math
from IPython.display import display

def get_metrics(model: str, optimizer: str, rho: float, seed: int, alpha: float = None) -> dict:
    is_baseline = alpha is None
    data = json.loads(METRICS_PATHS[model]["baseline" if is_baseline else "reparam"].read_text())

    for entry in data:
        summary = entry["summary"] if is_baseline else entry
        if summary["optimizer"] != optimizer or not math.isclose(summary["rho"], rho):
            continue
        if not is_baseline and not math.isclose(summary["alpha"], alpha):
            continue
        for s in entry["per_seed"]:
            if s["seed"] != seed:
                continue
            return dict(
                history=s["history"],
                train_loss=s["train_loss"],
                test_loss=s["test_loss"],
                divergence_rate=s.get("divergence_rate"),
                elapsed_sec=s["elapsed_sec"],
            )

    print(f"Metrics not found: model={model}, optimizer={optimizer}, rho={rho}, seed={seed}, alpha={alpha}")
    return None


def plot_epoch_vs_train_and_test_loss(model: str, optimizer: str, rho: float, seed: int, alpha: float = None):
    baseline_data = get_metrics(model, optimizer, rho, seed)
    reparam_data  = get_metrics(model, optimizer, rho, seed, alpha)

    traces = []

    if baseline_data is not None:
        baseline = pd.DataFrame(baseline_data["history"])
        if "train_loss" in baseline.columns:
            traces.append(go.Scatter(x=baseline["epoch"], y=baseline["train_loss"], mode="lines",
                       name="Baseline — Train", line=dict(color="#78d57e", dash="dash")))
        if "test_loss" in baseline.columns:
            traces.append(go.Scatter(x=baseline["epoch"], y=baseline["test_loss"], mode="lines",
                       name="Baseline — Test", line=dict(color="#78d57e")))


    if reparam_data is not None:
        reparam = pd.DataFrame(reparam_data["history"])
        if "train_loss" in reparam.columns:
            traces.append(go.Scatter(x=reparam["epoch"], y=reparam["train_loss"], mode="lines",
                       name=f"Reparam (α={alpha}) — Train", line=dict(color="#da655d", dash="dash")))
        if "test_loss" in reparam.columns:
            traces.append(go.Scatter(x=reparam["epoch"], y=reparam["test_loss"], mode="lines",
                       name=f"Reparam (α={alpha}) — Test", line=dict(color="#da655d")))

    if not traces:
        print(f"No data to plot for model={model}, optimizer={optimizer}, rho={rho}, seed={seed}, alpha={alpha}")
        return

    fig = go.FigureWidget(traces)
    fig.update_layout(
        title=f"{model} | {optimizer} | ρ={rho} | seed={seed} | α={alpha}",
        xaxis_title="Epoch", yaxis_title="Loss", height=450,
        legend=dict(orientation="v", xanchor="left", x=1.02, yanchor="middle", y=0.5),
    )
    display(fig)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

model_w     = widgets.Dropdown(options=["resnet18", "vit_b_32"], description="Model:")
optimizer_w = widgets.Dropdown(options=["sgd", "sam", "asam", "msam"], description="Optimizer:")
rho_w       = widgets.Dropdown(options=[0.0, 0.05, 0.1, 0.5], description="ρ:")
seed_w      = widgets.Dropdown(options=[40, 41, 42], value=42, description="Seed:")
alpha_w     = widgets.Dropdown(options=[5.0], value=5.0, description="α:")

ui = widgets.VBox([
    widgets.HBox([model_w, optimizer_w]),
    widgets.HBox([rho_w, seed_w, alpha_w]),
])

out = widgets.interactive_output(
    plot_epoch_vs_train_and_test_loss,
    dict(model=model_w, optimizer=optimizer_w, rho=rho_w, seed=seed_w, alpha=alpha_w),
)

display(ui, out)


Output()

### Pearson correlation — Sharpness vs Divergence Rate

We test whether the Hutchinson sharpness estimate $\text{tr}(H)/d$ is correlated with the divergence rate $\delta = \mathcal{L}_{test} - \mathcal{L}_{train}$ across all optimizer/ρ configurations.
Each point represents one (optimizer, ρ, α) config averaged over seeds.

In [8]:
from scipy import stats

SHARPNESS_PATHS = {
    "resnet18": RESULTS_DIR / "resnet18/experiments/flatness/resnet18/sharpness_all.json",
    "vit_b_32": RESULTS_DIR / "vit_b_32/experiments/flatness/vit_b_32/sharpness_all.json",
}


def build_sharpness_divergence_df(model: str) -> pd.DataFrame:
    sharpness    = json.loads(SHARPNESS_PATHS[model].read_text())
    reparam_data = json.loads(METRICS_PATHS[model]["reparam"].read_text())

    div_lookup = {
        (e["optimizer"], e["rho"], e["alpha"]): float(np.mean([s["test_loss"] - s["train_loss"] for s in e["per_seed"]]))
        for e in reparam_data
    }

    rows = []
    for key, sharp_val in sharpness.items():
        if "_alpha" not in key:
            continue
        opt, rest   = key.split("_rho", 1)
        rho_str, alpha_str = rest.split("_alpha", 1)
        div_val = div_lookup.get((opt, float(rho_str), float(alpha_str)))
        # if div_val is None:
        #     continue
        rows.append(dict(model=model, optimizer=opt, rho=float(rho_str), alpha=float(alpha_str),
                         sharpness=sharp_val, divergence_rate=div_val))
    return pd.DataFrame(rows)


def plot_pearson_sharpness_vs_divergence(models: list) -> None:
    dfs = []
    for m in models:
        try:
            dfs.append(build_sharpness_divergence_df(m))
        except FileNotFoundError as e:
            print(f"Skipping {m}: {e}")
    if not dfs:
        print("No data found.")
        return

    df = pd.concat(dfs, ignore_index=True)
    r, p = stats.pearsonr(df["sharpness"], df["divergence_rate"])

    OPT_COLOR    = {"sgd": "#9E9E9E", "sam": "#2196F3", "asam": "#FF9800", "msam": "#4CAF50"}
    MODEL_SYMBOL = {"resnet18": "circle", "vit_b_32": "diamond"}

    fig = go.Figure()
    for (opt, mdl), grp in df.groupby(["optimizer", "model"]):
        fig.add_trace(go.Scatter(
            x=grp["sharpness"], y=grp["divergence_rate"], mode="markers",
            name=f"{opt.upper()} / {mdl}",
            marker=dict(color=OPT_COLOR.get(opt, "gray"), symbol=MODEL_SYMBOL.get(mdl, "circle"),
                        size=10, line=dict(width=1, color="white")),
            text=[f"ρ={row.rho}" for row in grp.itertuples()],
            hovertemplate="<b>%{text}</b><br>Sharpness: %{x:.2e}<br>Divergence: %{y:.4f}<extra>%{fullData.name}</extra>",
        ))

    x_range = np.linspace(df["sharpness"].min(), df["sharpness"].max(), 100)
    slope, intercept, *_ = stats.linregress(df["sharpness"], df["divergence_rate"])
    fig.add_trace(go.Scatter(
        x=x_range.tolist(), y=(slope * x_range + intercept).tolist(),
        mode="lines", name="OLS fit", line=dict(color="black", dash="dash", width=1),
    ))

    fig.update_layout(
        title=f"Sharpness vs Divergence Rate  |  Pearson r = {r:.3f}  (p = {p:.3e})",
        xaxis_title="Sharpness  tr(H)/d", yaxis_title="Divergence rate  (L_test − L_train)",
        height=500, template="plotly_white",
        legend=dict(orientation="v", xanchor="left", x=1.02, yanchor="middle", y=0.5),
    )
    fig.show()
    print(f"Pearson r = {r:.4f},  p-value = {p:.4e},  n = {len(df)}")


plot_pearson_sharpness_vs_divergence(["resnet18", "vit_b_32"])


Pearson r = 0.5909,  p-value = 7.2019e-02,  n = 10
